# 11 — Random Forest Baselines

Full ablation set: RNA-only, protein-only, RNA+drug, protein+drug. Fixed hyperparameters, top-variance feature selection computed from train-fold cell lines only (no leakage), evaluated on all 4 test splits per fold with `evaluateMT`.

## Environment setup (Colab or local)

In [2]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")


Mounted at /content/drive
Running on Colab | BASE_PATH = /content/drive/MyDrive/multiomics_project


## Imports and config

In [3]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import r2_score, roc_auc_score
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator


In [11]:
DATA_DIR = BASE_PATH / "data" / "GDSC2"
PROCESSED_DIR = BASE_PATH / "data" / "processed"
SPLITS_DIR = BASE_PATH / "data" / "splits"
RESULTS_DIR = BASE_PATH / "results" / "random_forest"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

SPLIT_TYPES = ["lpo", "lco", "ldo", "lto"]
TOP_K_FEATURES = 1000
RANDOM_STATE = 42

RF_PARAMS = dict(n_estimators=500, max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE)


## Load response pairs + splits (from notebook 04)

In [5]:
pairs = pd.read_parquet(PROCESSED_DIR / "response_pairs.parquet")

with open(SPLITS_DIR / "splits.json") as f:
    folds = json.load(f)

with open(SPLITS_DIR / "holdout_groups.json") as f:
    holdout_groups = json.load(f)

print(f"{len(pairs)} pairs loaded")
for fd in folds:
    print(f"fold {fd['fold']}: train={len(fd['train']):,} | "
          f"lco_test={len(fd['lco_test']):,} | ldo_test={len(fd['ldo_test']):,} | "
          f"lto_test={len(fd['lto_test']):,} | lpo_test={len(fd['lpo_test']):,}")


176197 pairs loaded
fold 0: train=107,421 | lco_test=17,470 | ldo_test=18,404 | lto_test=25,964 | lpo_test=20,613
fold 1: train=118,444 | lco_test=17,331 | ldo_test=18,515 | lto_test=13,579 | lpo_test=16,475
fold 2: train=97,262 | lco_test=18,173 | ldo_test=18,126 | lto_test=35,198 | lpo_test=23,832
fold 3: train=111,375 | lco_test=17,849 | ldo_test=18,140 | lto_test=21,451 | lpo_test=19,147
fold 4: train=103,126 | lco_test=17,762 | ldo_test=18,533 | lto_test=28,869 | lpo_test=21,721


## Load omics + drug fingerprints

Dedupe `cellosaurus_id` immediately after load (both RNA and protein have known duplicate rows). Protein gets `fillna(0)` for missing values -- BDRN-validated choice.

In [16]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")

rna = rna[~rna.index.duplicated(keep="first")].iloc[:, 1:]
protein = protein[~protein.index.duplicated(keep="first")].fillna(0)

protein_aligned = pd.read_parquet(PROCESSED_DIR / "proteomics_gene_symbol_aligned.parquet")
protein_aligned = protein_aligned[~protein_aligned.index.duplicated(keep="first")].fillna(0)
print(f"Protein (gene-symbol aligned): {protein_aligned.shape}")

OMICS = {"rna": rna, "protein": protein, "protein_aligned": protein_aligned}

print(f"RNA: {rna.shape[1]} genes / {rna.shape[0]} cell lines")
print(f"Protein: {protein.shape[1]} proteins / {protein.shape[0]} cell lines")


Protein (gene-symbol aligned): (860, 6667)
RNA: 17737 genes / 1010 cell lines
Protein: 6692 proteins / 860 cell lines


In [7]:
def build_drug_fingerprints(drug_smiles_df: pd.DataFrame, radius: int = 2, n_bits: int = 2048) -> dict:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        if mol is None:
            continue
        fp = generator.GetFingerprint(mol)
        fps[row[COL_DRUG]] = np.array(fp, dtype=np.float32)
    return fps


drug_fp = build_drug_fingerprints(drug_smiles)
print(f"Fingerprints: {len(drug_fp)} / {drug_smiles[COL_DRUG].nunique()} drugs")


Fingerprints: 246 / 246 drugs


## Evaluation metric

Same `evaluateMT` used in notebooks 09 and 10 -- NaN-safe for constant-prediction cases (e.g. a naive fallback, or a degenerate RF leaf).

In [8]:
def evaluateMT(target, pred, threshold=None):
    """
    threshold: cutoff for binarizing target into sensitive/resistant for ROC-AUC.
    Defaults to median of target if not provided.
    """
    variance_real = np.var(target)
    variance_pred = np.var(pred)
    mses = ((target - pred) ** 2).mean(axis=0)
    rmse = np.sqrt(mses)

    pred_is_constant = np.isclose(variance_pred, 0)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if pred_is_constant:
            correlation, corr_p_value = np.nan, np.nan
            spearman_corr, spearman_p = np.nan, np.nan
            slope, intercept, r_value, lr_p_value, std_err = (np.nan,) * 5
        else:
            correlation, corr_p_value = pearsonr(target, pred)
            spearman_corr, spearman_p = spearmanr(target, pred)
            slope, intercept, r_value, lr_p_value, std_err = linregress(pred, target)

    r2 = r2_score(target, pred)

    if threshold is None:
        threshold = np.median(target)
    target_binary = (target < threshold).astype(int)
    if len(np.unique(target_binary)) == 2 and not pred_is_constant:
        roc_auc = roc_auc_score(target_binary, -pred)
    else:
        roc_auc = np.nan

    results = {
        'Correlation': round(correlation, 2) if not np.isnan(correlation) else correlation,
        'Corr p-value': round(corr_p_value, 4) if not np.isnan(corr_p_value) else corr_p_value,
        'Spearman': round(spearman_corr, 2) if not np.isnan(spearman_corr) else spearman_corr,
        'MSE': round(mses, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 2),
        'ROC-AUC': round(roc_auc, 2) if not np.isnan(roc_auc) else roc_auc,
        'Slope': round(slope, 2) if not np.isnan(slope) else slope,
        'Standard error': round(std_err, 2) if not np.isnan(std_err) else std_err,
        'Variance Real': round(variance_real, 2),
        'Variance Pred': round(variance_pred, 2)
    }
    return pd.DataFrame([results])


## Feature selection + matrix builders

Top-variance feature selection is computed from **train-fold cell lines only** (not the full omics table) to avoid any leakage from val/test cell lines into the selected gene/protein set. Selection happens on the compact per-cell-line table, before expanding to per-pair rows -- expanding first causes the ~11GB memory blowup hit in earlier sessions.

In [17]:
def select_top_variance_cols(train_idx: np.ndarray, arm: str, k: int = TOP_K_FEATURES) -> pd.Index:
    """Top-k variance columns for an omics arm, computed from train-fold cell lines only."""
    train_cells = pairs.loc[train_idx, COL_CELLOSAURUS].unique()
    compact = OMICS[arm].loc[OMICS[arm].index.intersection(train_cells)]
    return compact.var(axis=0).sort_values(ascending=False).index[:k]

def select_residual_variance_cols(
    train_idx: np.ndarray,
    protein_aligned: pd.DataFrame,   # protein with gene-symbol columns (from notebook 02)
    k: int = TOP_K_FEATURES,
) -> pd.Index:
    """
    Select top-k proteins by residual variance after regressing protein on matched RNA.
    Proteins where RNA explains little of the variance rank highest — these carry
    signal independent of RNA. Unmatched proteins (no RNA counterpart) use raw variance.
    Computed from train-fold cell lines only.
    """
    train_cells = pairs.loc[train_idx, COL_CELLOSAURUS].unique()
    matched_cells = protein_aligned.index.intersection(train_cells)

    rna_sub     = rna.loc[rna.index.intersection(matched_cells)]
    protein_sub = protein_aligned.loc[matched_cells]
    common_genes = rna_sub.columns.intersection(protein_sub.columns)

    residual_vars = {}
    for gene in protein_sub.columns:
        x = rna_sub[gene].values if gene in common_genes else None
        y = protein_sub[gene].values

        if x is not None:
            mask = ~np.isnan(x) & ~np.isnan(y)
            if mask.sum() >= 10:
                slope, intercept = np.polyfit(x[mask], y[mask], 1)
                residual = y[mask] - (slope * x[mask] + intercept)
                residual_vars[gene] = np.var(residual)
                continue
        # unmatched or too few points — fall back to raw variance
        residual_vars[gene] = np.nanvar(y)

    ranked = pd.Series(residual_vars).sort_values(ascending=False)
    # return as index into the original protein column names (before gene-symbol alignment)
    return ranked.index[:k]


def build_rf_matrix(idx: np.ndarray, components: list[str], top_cols: dict) -> tuple[np.ndarray, np.ndarray]:
    sub = pairs.loc[idx]
    parts = []
    if "rna" in components:
        parts.append(OMICS["rna"][top_cols["rna"]].loc[sub[COL_CELLOSAURUS]].to_numpy())
    if "protein" in components:
        # use gene-symbol-aligned table if top_cols["protein"] contains gene symbols
        # (i.e. residual-variance selection), otherwise use raw protein
        protein_table = (
            OMICS["protein_aligned"]
            if top_cols["protein"].isin(protein_aligned.columns).all()
            else OMICS["protein"]
        )
        parts.append(protein_table[top_cols["protein"]].loc[sub[COL_CELLOSAURUS]].to_numpy())
    if "drug" in components:
        parts.append(np.vstack([drug_fp[d] for d in sub[COL_DRUG]]))

    X = np.hstack(parts).astype(np.float32)
    y = sub[COL_IC50].to_numpy().astype(np.float32)

    assert not np.isnan(X).any(), f"NaNs in features for components={components}"
    assert not np.isnan(y).any(), "NaNs in target"

    return X, y


## RandomForest fit + evaluate, per arm

One RandomForest is fit per fold (on the shared train set), then evaluated on all 4 test splits from that same fold -- consistent with the new `splits.json` structure (one train, four tests).

In [13]:
def run_rf_arm(
    arm_name: str,
    components: list[str],
    protein_selector=None,  # if None, falls back to top-variance
) -> list[pd.DataFrame]:
    """Fits one RandomForest per fold for this arm, evaluates on all 4 test splits.
    protein_selector: callable(train_idx) -> pd.Index, or None for default top-variance.
    """
    results = []
    for fold_i, fold in enumerate(folds):
        start = time.time()
        train_idx = np.array(fold["train"])

        top_cols = {}
        if "rna" in components:
            top_cols["rna"] = select_top_variance_cols(train_idx, "rna")
        if "protein" in components:
            if protein_selector is not None:
                top_cols["protein"] = protein_selector(train_idx)
            else:
                top_cols["protein"] = select_top_variance_cols(train_idx, "protein")

        X_train, y_train = build_rf_matrix(train_idx, components, top_cols)

        model = RandomForestRegressor(**RF_PARAMS)
        model.fit(X_train, y_train)

        for split_type in SPLIT_TYPES:
            test_idx = np.array(fold[f"{split_type}_test"])
            X_test, y_test = build_rf_matrix(test_idx, components, top_cols)
            y_pred = model.predict(X_test)

            df = evaluateMT(y_test, y_pred)
            df.insert(0, "Fold", fold_i)
            df.insert(1, "Split", split_type.upper())
            df.insert(2, "Arm", arm_name)
            results.append(df)

        elapsed = time.time() - start
        print(f"[{arm_name}] fold {fold_i}: trained on {len(train_idx):,} pairs, "
              f"{len(components)} component(s) -- {elapsed:.1f}s")

    return results

### RNA-only

In [10]:
rna_results = run_rf_arm("RNA", ["rna"])

[RNA] fold 0: trained on 107,421 pairs, 1 component(s) -- 105.6s
[RNA] fold 1: trained on 118,444 pairs, 1 component(s) -- 117.5s
[RNA] fold 2: trained on 97,262 pairs, 1 component(s) -- 96.6s
[RNA] fold 3: trained on 111,375 pairs, 1 component(s) -- 110.9s
[RNA] fold 4: trained on 103,126 pairs, 1 component(s) -- 101.8s


### Drug-only

In [12]:
drug_results = run_rf_arm("Drug", ["drug"])

[Drug] fold 0: trained on 107,421 pairs, 1 component(s) -- 176.2s
[Drug] fold 1: trained on 118,444 pairs, 1 component(s) -- 188.3s
[Drug] fold 2: trained on 97,262 pairs, 1 component(s) -- 156.9s
[Drug] fold 3: trained on 111,375 pairs, 1 component(s) -- 184.2s
[Drug] fold 4: trained on 103,126 pairs, 1 component(s) -- 168.8s


### Protein-only

In [11]:
protein_results = run_rf_arm("Protein", ["protein"])

[Protein] fold 0: trained on 107,421 pairs, 1 component(s) -- 89.4s
[Protein] fold 1: trained on 118,444 pairs, 1 component(s) -- 99.4s
[Protein] fold 2: trained on 97,262 pairs, 1 component(s) -- 81.9s
[Protein] fold 3: trained on 111,375 pairs, 1 component(s) -- 96.0s
[Protein] fold 4: trained on 103,126 pairs, 1 component(s) -- 87.4s


### Protein-only (residual-variance selection)

In [18]:
protein_rv_results = run_rf_arm(
    "Protein-RV",
    ["protein"],
    protein_selector=lambda train_idx: select_residual_variance_cols(train_idx, protein_aligned),
)

[Protein-RV] fold 0: trained on 107,421 pairs, 1 component(s) -- 103.9s
[Protein-RV] fold 1: trained on 118,444 pairs, 1 component(s) -- 114.7s
[Protein-RV] fold 2: trained on 97,262 pairs, 1 component(s) -- 94.5s
[Protein-RV] fold 3: trained on 111,375 pairs, 1 component(s) -- 106.0s
[Protein-RV] fold 4: trained on 103,126 pairs, 1 component(s) -- 100.3s


In [19]:
protein_rv_res_df = pd.concat(protein_rv_results)

In [20]:
protein_rv_res_df

,Fold,Split,Arm,Correlation,Corr p-value,Spearman,MSE,RMSE,R2,ROC-AUC,Slope,Standard error,Variance Real,Variance Pred
0,0,LPO,Protein-RV,0.31,0.0,0.34,6.55,2.56,0.10,0.66,1.01,0.02,7.27,0.70
0,0,LCO,Protein-RV,0.21,0.0,0.22,6.84,2.61,0.05,0.61,1.02,0.04,7.16,0.32
0,0,LDO,Protein-RV,0.31,0.0,0.35,5.89,2.43,0.10,0.67,0.96,0.02,6.52,0.69
0,0,LTO,Protein-RV,0.25,0.0,0.24,7.14,2.67,0.05,0.61,1.26,0.03,7.54,0.29
0,1,LPO,Protein-RV,0.30,0.0,0.32,6.67,2.58,0.09,0.65,0.94,0.02,7.33,0.76
0,1,LCO,Protein-RV,0.22,0.0,0.25,6.91,2.63,0.04,0.61,1.05,0.04,7.23,0.32
0,1,LDO,Protein-RV,0.32,0.0,0.35,6.59,2.57,0.10,0.66,1.00,0.02,7.34,0.75
0,1,LTO,Protein-RV,0.16,0.0,0.17,6.70,2.59,0.02,0.58,1.57,0.09,6.84,0.07
0,2,LPO,Protein-RV,0.30,0.0,0.33,6.54,2.56,0.09,0.66,1.00,0.02,7.20,0.66
0,2,LCO,Protein-RV,0.24,0.0,0.29,6.94,2.63,0.06,0.63,1.29,0.04,7.34,0.26


### RNA + drug fingerprint

In [18]:
rna_drug_results = run_rf_arm("RNA+Drug", ["rna", "drug"])

[RNA+Drug] fold 0: trained on 107,421 pairs, 2 component(s) -- 376.4s
[RNA+Drug] fold 1: trained on 118,444 pairs, 2 component(s) -- 407.8s
[RNA+Drug] fold 2: trained on 97,262 pairs, 2 component(s) -- 335.7s
[RNA+Drug] fold 3: trained on 111,375 pairs, 2 component(s) -- 376.7s
[RNA+Drug] fold 4: trained on 103,126 pairs, 2 component(s) -- 355.6s


### Protein + drug fingerprint

In [10]:
protein_drug_results = run_rf_arm("Protein+Drug", ["protein", "drug"])

[Protein+Drug] fold 0: trained on 107,421 pairs, 2 component(s) -- 366.0s
[Protein+Drug] fold 1: trained on 118,444 pairs, 2 component(s) -- 396.4s
[Protein+Drug] fold 2: trained on 97,262 pairs, 2 component(s) -- 335.3s
[Protein+Drug] fold 4: trained on 103,126 pairs, 2 component(s) -- 355.4s


In [20]:
prot_res_df = pd.concat(protein_drug_results)

### RNA + protein + drug 

In [19]:
rna_protein_drug_results = run_rf_arm("RNA+Protein+Drug", ["rna", "protein", "drug"])

[RNA+Protein+Drug] fold 0: trained on 107,421 pairs, 3 component(s) -- 556.5s
[RNA+Protein+Drug] fold 1: trained on 118,444 pairs, 3 component(s) -- 612.0s
[RNA+Protein+Drug] fold 2: trained on 97,262 pairs, 3 component(s) -- 508.0s
[RNA+Protein+Drug] fold 3: trained on 111,375 pairs, 3 component(s) -- 563.3s
[RNA+Protein+Drug] fold 4: trained on 103,126 pairs, 3 component(s) -- 548.2s


In [21]:

rna_prot_drug_res_df = pd.concat(rna_protein_drug_results)

## Combine results

In [25]:
rf_results_df = pd.concat(
    protein_drug_results + rna_protein_drug_results,
    ignore_index=True
)
rf_results_df

,Fold,Split,Arm,Correlation,Corr p-value,Spearman,MSE,RMSE,R2,ROC-AUC,Slope,Standard error,Variance Real,Variance Pred
0,0,LPO,Protein+Drug,0.86,0.0,0.83,1.85,1.36,0.74,0.91,1.06,0.00,7.27,4.82
1,0,LCO,Protein+Drug,0.84,0.0,0.79,2.14,1.46,0.70,0.89,1.03,0.01,7.16,4.72
2,0,LDO,Protein+Drug,0.32,0.0,0.35,5.87,2.42,0.10,0.67,0.99,0.02,6.52,0.69
3,0,LTO,Protein+Drug,0.85,0.0,0.81,2.19,1.48,0.71,0.90,1.05,0.00,7.54,4.91
4,1,LPO,Protein+Drug,0.85,0.0,0.81,2.01,1.42,0.73,0.90,1.05,0.00,7.33,4.84
5,1,LCO,Protein+Drug,0.82,0.0,0.77,2.37,1.54,0.67,0.88,1.01,0.01,7.23,4.75
6,1,LDO,Protein+Drug,0.22,0.0,0.22,7.15,2.67,0.03,0.58,0.65,0.02,7.34,0.84
7,1,LTO,Protein+Drug,0.81,0.0,0.76,2.37,1.54,0.65,0.87,1.02,0.01,6.84,4.29
8,2,LPO,Protein+Drug,0.88,0.0,0.83,1.66,1.29,0.77,0.90,1.06,0.00,7.20,4.97
9,2,LCO,Protein+Drug,0.86,0.0,0.80,1.97,1.40,0.73,0.89,1.04,0.00,7.34,4.95


In [ ]:
rf_results_df = pd.concat(
    rna_results + protein_results + drug_results + rna_drug_results + protein_drug_results,
    ignore_index=True
)
rf_results_df


,Fold,Split,Arm,Correlation,Corr p-value,Spearman,MSE,RMSE,R2,ROC-AUC,Slope,Standard error,Variance Real,Variance Pred
0,0,LPO,RNA,0.31,0.0,0.34,6.56,2.56,0.10,0.66,1.02,0.02,7.27,0.68
1,0,LCO,RNA,0.21,0.0,0.22,6.86,2.62,0.04,0.61,1.08,0.04,7.16,0.27
2,0,LDO,RNA,0.31,0.0,0.35,5.90,2.43,0.10,0.67,0.96,0.02,6.52,0.67
3,0,LTO,RNA,0.24,0.0,0.24,7.20,2.68,0.05,0.61,1.30,0.03,7.54,0.26
4,1,LPO,RNA,0.30,0.0,0.32,6.67,2.58,0.09,0.65,0.94,0.02,7.33,0.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,3,LTO,RNA+Drug,0.81,0.0,0.78,2.45,1.56,0.65,0.88,1.04,0.01,7.03,4.22
76,4,LPO,RNA+Drug,0.86,0.0,0.83,1.92,1.38,0.73,0.91,1.07,0.00,7.21,4.63
77,4,LCO,RNA+Drug,0.85,0.0,0.81,2.07,1.44,0.71,0.90,1.07,0.00,7.23,4.59
78,4,LDO,RNA+Drug,0.40,0.0,0.43,5.62,2.37,0.16,0.71,1.03,0.02,6.65,0.98


## Summary: mean across folds, per arm x split

In [21]:
def mean_std_str(s: pd.Series) -> str:
    return f"{s.mean():.3f} ± {s.std():.3f}"

summary_df = (
    protein_rv_res_df
    .groupby(["Arm", "Split"])[["Correlation", "Spearman", "RMSE", "R2", "ROC-AUC"]]
    .agg(mean_std_str)
    .reset_index()
)
summary_df

,Arm,Split,Correlation,Spearman,RMSE,R2,ROC-AUC
0,Protein-RV,LCO,0.230 ± 0.014,0.252 ± 0.026,2.614 ± 0.021,0.052 ± 0.008,0.618 ± 0.008
1,Protein-RV,LDO,0.320 ± 0.042,0.340 ± 0.037,2.446 ± 0.393,0.066 ± 0.076,0.660 ± 0.023
2,Protein-RV,LPO,0.306 ± 0.009,0.332 ± 0.008,2.558 ± 0.015,0.094 ± 0.005,0.656 ± 0.005
3,Protein-RV,LTO,0.222 ± 0.048,0.230 ± 0.048,2.638 ± 0.036,0.046 ± 0.021,0.606 ± 0.021


In [25]:
def mean_std_str(s: pd.Series) -> str:
    return f"{s.mean():.3f} ± {s.std():.3f}"

summary_df = (
    rf_results_df
    .groupby(["Arm", "Split"])[["Correlation", "Spearman", "RMSE", "R2", "ROC-AUC"]]
    .agg(mean_std_str)
    .reset_index()
)
summary_df


,Arm,Split,Correlation,Spearman,RMSE,R2,ROC-AUC
0,Drug,LCO,0.790 ± 0.020,0.726 ± 0.018,1.652 ± 0.053,0.618 ± 0.026,0.858 ± 0.011
1,Drug,LDO,0.104 ± 0.155,0.094 ± 0.160,2.566 ± 0.380,-0.028 ± 0.056,0.528 ± 0.086
2,Drug,LPO,0.788 ± 0.015,0.724 ± 0.009,1.660 ± 0.047,0.616 ± 0.022,0.854 ± 0.005
3,Drug,LTO,0.794 ± 0.011,0.738 ± 0.008,1.768 ± 0.131,0.570 ± 0.047,0.862 ± 0.004
4,Protein,LCO,0.232 ± 0.011,0.250 ± 0.022,2.616 ± 0.023,0.050 ± 0.007,0.618 ± 0.008
5,Protein,LDO,0.320 ± 0.042,0.340 ± 0.037,2.446 ± 0.393,0.066 ± 0.076,0.660 ± 0.023
6,Protein,LPO,0.306 ± 0.009,0.330 ± 0.010,2.558 ± 0.015,0.094 ± 0.005,0.656 ± 0.005
7,Protein,LTO,0.224 ± 0.045,0.232 ± 0.045,2.638 ± 0.036,0.044 ± 0.018,0.606 ± 0.021
8,RNA,LCO,0.224 ± 0.013,0.244 ± 0.015,2.622 ± 0.026,0.046 ± 0.009,0.618 ± 0.004
9,RNA,LDO,0.320 ± 0.042,0.340 ± 0.037,2.450 ± 0.396,0.066 ± 0.076,0.660 ± 0.023


## Save results

In [26]:
rf_results_df.to_parquet(RESULTS_DIR / "random_forest_single_modality_per_fold.parquet")
summary_df.to_csv(RESULTS_DIR / "random_forest_single_modality_rnadrug_summary.csv", index=False)
print(f"Saved to {RESULTS_DIR}")


Saved to /content/drive/MyDrive/multiomics_project/results/random_forest


In [27]:
rf_results_df.to_parquet(RESULTS_DIR / "random_forest_protein_drug_3_per_fold.parquet")
summary_df.to_csv(RESULTS_DIR / "random_forest_protein_drug_3_summary.csv", index=False)
print(f"Saved to {RESULTS_DIR}")

Saved to /content/drive/MyDrive/multiomics_project/results/random_forest
